# Ước lượng giá rao bán nhà ở Hà Nội

**Dataset:** `locngodev/vietnam-real-estate` — 1 file `raw_data_all.csv`, ~3,5 triệu dòng, 19 cột.

Notebook này chạy **trên Kaggle**: chỉ cần *Add Input* dataset, không tải 3,2 GB về máy.

**Phạm vi:** `property_type_name == "Nhà"` và `province_name == "Hà Nội"`.
**Mục tiêu dự đoán:** `price` — đây là **giá rao bán**, không phải giá giao dịch thực tế.

## 0. Môi trường và đường dẫn dữ liệu

In [ ]:
import glob, hashlib, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
if not os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and not Path("/kaggle").exists():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.width", 140)

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)

override = os.environ.get("VIETNAM_RAW")
cands = [override] if override else sorted(glob.glob("/kaggle/input/**/raw_data_all.csv", recursive=True))
assert cands, "Chưa gắn dataset: panel phải → Add Input → locngodev/vietnam-real-estate"
RAW = Path(cands[0])
print("RAW     :", RAW)
print("Kích cỡ :", f"{RAW.stat().st_size / 1e9:.2f} GB")
print("Python  :", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

## 1. Quét toàn file theo chunk — đo "phễu" lọc dữ liệu

File 3,2 GB không đọc hết vào RAM. Đọc từng chunk 200.000 dòng, chỉ giữ các dòng thuộc phạm vi.
Đếm số dòng còn lại sau **từng bước lọc** để báo cáo trung thực.

In [ ]:
COLS = ["name", "property_type_name", "province_name", "district_name", "ward_name",
        "street_name", "project_name", "price", "area", "floor_count",
        "frontage_width", "bedroom_count", "bathroom_count", "house_direction",
        "published_at"]
CHUNK = 200_000

funnel = {"doc": 0, "nha": 0, "ha_noi": 0, "dien_tich_gia_hop_le": 0}
parts = []

for chunk in pd.read_csv(RAW, usecols=COLS, chunksize=CHUNK, dtype="string"):
    funnel["doc"] += len(chunk)
    step = chunk[chunk.property_type_name.str.strip() == "Nhà"].copy()
    funnel["nha"] += len(step)
    step = step[step.province_name.str.strip() == "Hà Nội"].copy()
    funnel["ha_noi"] += len(step)
    for c in ["price", "area", "floor_count", "frontage_width", "bedroom_count", "bathroom_count"]:
        step[c] = pd.to_numeric(step[c], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
    step = step[step.area.between(10, 500) & step.price.gt(0) & step.name.notna()]
    funnel["dien_tich_gia_hop_le"] += len(step)
    if len(step):
        parts.append(step)

if not parts:
    raise ValueError("Không có dòng phù hợp phạm vi Hà Nội + Nhà; kiểm tra dataset/nhãn.")
df = pd.concat(parts, ignore_index=True)
pd.DataFrame([funnel]).T.rename(columns={0: "rows"}).to_csv(OUT / "funnel.csv")
print(json.dumps(funnel, indent=2, ensure_ascii=False))
print(f"\nGiữ lại {funnel['dien_tich_gia_hop_le']/funnel['doc']:.2%} tổng số dòng.")

## 2. Chuẩn hoá, chống trùng tin, tạo feature

Cùng một căn thường được đăng lại nhiều ngày. Dataset không có ID bất động sản, nên dùng
**hash(tỉnh + quận + tiêu đề chuẩn hoá)** làm *proxy nhóm* để chia train/test không chồng lấn.
Đây là proxy — tin sửa tiêu đề vẫn có thể lọt.

In [ ]:
def norm_text(s):
    return s.astype("string").str.normalize("NFC").str.strip()

for c in ["district_name", "ward_name", "street_name", "project_name",
          "house_direction", "property_type_name", "province_name"]:
    df[c] = norm_text(df[c]).replace("", pd.NA)

df["published_at"] = pd.to_datetime(df.published_at, errors="coerce", format="mixed")
df["month"] = df.published_at.dt.month
df["year"] = df.published_at.dt.year

# giá/m2 chỉ để AUDIT, không đưa vào feature
df["price_per_m2"] = df.price / df.area

keys = (df.province_name.fillna("") + "|" + df.district_name.fillna("") + "|"
        + df.name.str.lower().str.replace(r"\s+", " ", regex=True))
df["group_id"] = keys.map(lambda s: hashlib.sha256(s.encode()).hexdigest())

before = len(df)
df = df.drop_duplicates(subset=["group_id", "price", "area", "district_name"])
df = df.reset_index(drop=True)

clean = df[["district_name", "ward_name", "area", "bedroom_count", "bathroom_count",
            "frontage_width", "price", "price_per_m2", "month", "year", "group_id"]]
clean.to_parquet(OUT / "clean.parquet", index=False)

print(f"Trước dedupe: {before:,} | sau: {len(df):,} | nhóm tiêu đề: {df.group_id.nunique():,}")
print("\nTỉ lệ thiếu:")
print(clean.isna().mean().round(3).to_string())
del parts, chunk, step
import gc; gc.collect()
display(clean.describe(percentiles=[.01, .5, .99]).round(1))

## 3. EDA — bốn thư viện: Pandas, NumPy, Matplotlib, Seaborn

### 3.1 Phân phối giá và diện tích

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))

sns.histplot(df.price / 1e9, bins=80, ax=ax[0, 0], color="#ff6a41")
ax[0, 0].set(title="Phân phối giá rao bán (tỷ VND)", xlabel="Giá (tỷ VND)")
ax[0, 0].set_xlim(0, np.percentile(df.price / 1e9, 99))

sns.histplot(np.log1p(df.price), bins=80, ax=ax[0, 1], color="#2f6fed")
ax[0, 1].set(title="Phân phối log1p(price)", xlabel="log1p(price)")

sns.histplot(df.area, bins=60, ax=ax[1, 0], color="#7a5af8")
ax[1, 0].set(title="Phân phối diện tích", xlabel="Diện tích (m²)")

sns.histplot(df.price_per_m2 / 1e6, bins=80, ax=ax[1, 1], color="#12b76a")
ax[1, 1].set(title="Giá trên mỗi m² (triệu/m²)", xlabel="Triệu VND/m²")
ax[1, 1].set_xlim(0, np.percentile(df.price_per_m2 / 1e6, 99))

plt.tight_layout()
plt.savefig(OUT / "fig1_distributions.png", dpi=130, bbox_inches="tight")
plt.show()

print(df[["price", "area", "price_per_m2"]].describe(percentiles=[.001, .01, .5, .99]).round(0).to_string())

**Cách đọc:** so sánh đuôi phân phối trước/sau log. Không mặc định dữ liệu đã phân phối chuẩn; chỉ kết luận sau khi xem kết quả. Giá trị cực đoan chưa được xác minh nên có thể làm sai lệch đồ thị và metric.

### 3.2 Quan hệ diện tích – giá

In [ ]:
samp = df.sample(min(20_000, len(df)), random_state=42)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(samp.area, samp.price / 1e9, s=6, alpha=.25, color="#ff6a41")
ax[0].set(title="Diện tích vs giá (mẫu 20k tin)", xlabel="Diện tích (m²)", ylabel="Giá (tỷ VND)")

ax[1].scatter(samp.area, np.log1p(samp.price), s=6, alpha=.25, color="#2f6fed")
ax[1].set(title="Cùng dữ liệu ở thang log", xlabel="Diện tích (m²)", ylabel="log1p(price)")

plt.tight_layout()
plt.savefig(OUT / "fig2_area_price.png", dpi=130, bbox_inches="tight")
plt.show()

print("Tương quan Pearson  area – price     :", round(df.area.corr(df.price), 3))
print("Tương quan Pearson  area – log(price) :", round(df.area.corr(np.log1p(df.price)), 3))
print("Tương quan Spearman area – price     :", round(df.area.corr(df.price, method="spearman"), 3))

### 3.3 Giá theo quận và tương quan giữa các biến số

In [ ]:
top = df.district_name.value_counts().head(10).index
sub = df[df.district_name.isin(top)]

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=sub, x=sub.price / 1e9, y="district_name", order=top, ax=ax[0],
            color="#ff6a41", showfliers=False)
ax[0].set(title="Giá rao bán theo quận (bỏ outlier khỏi hình)", xlabel="Giá (tỷ VND)", ylabel="")
ax[0].invert_yaxis()

num = ["price", "area", "price_per_m2", "bedroom_count", "bathroom_count", "frontage_width"]
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, ax=ax[1])
ax[1].set_title("Ma trận tương quan")

plt.tight_layout()
plt.savefig(OUT / "fig3_district_corr.png", dpi=130, bbox_inches="tight")
plt.show()

med = (df.groupby("district_name").price.median().div(1e9).sort_values(ascending=False))
print("Trung vị giá theo quận (tỷ VND), top 10:")
print(med.head(10).round(2).to_string())

**Cảnh báo diễn giải:** `price_per_m2` tương quan rất cao với `price` vì nó *được tính từ* `price`.
Biến này **không được** đưa vào mô hình — chỉ dùng để kiểm tra outlier.
Tương quan không chứng minh quan hệ nhân quả.

## 4. Mô hình

- Chia tập theo **nhóm tiêu đề** (`GroupShuffleSplit`) — cùng một tin không nằm cả hai phía.
- Baseline: dự đoán trung vị, để biết mô hình có hơn "đoán bừa" hay không.
- `LinearRegression`, và `Ridge` với target `log1p`.
- Đo trên **VND gốc**: MAE, RMSE, MdAPE; R² báo trên thang giá gốc. Giá trị âm có thể xảy ra nếu mô hình kém mốc dự đoán trung bình.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM = ["area", "bedroom_count", "bathroom_count", "frontage_width"]
CAT = ["district_name", "ward_name"]

X = df[NUM + CAT].copy()
for c in CAT:
    X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
y = df.price.to_numpy(float)

assert df.group_id.nunique() >= 5, "Cần ít nhất 5 nhóm tin hợp lệ để chia tập."
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=42)
              .split(X, y, groups=df.group_id))
assert set(df.group_id.iloc[tr]).isdisjoint(df.group_id.iloc[te])
print(f"train {len(tr):,} dòng | test {len(te):,} dòng | {df.group_id.iloc[tr].nunique():,} nhóm train")


def build(est, log_target):
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True,
                                                keep_empty_features=True)),
                          ("sc", StandardScaler())]), NUM),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])
    pipe = Pipeline([("prep", prep), ("model", est)])
    return (TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
            if log_target else pipe)


models = {
    "Baseline (trung vị)": build(DummyRegressor(strategy="median"), False),
    "LinearRegression": build(LinearRegression(), False),
    "Ridge + log(target)": build(Ridge(alpha=10.0, solver="lsqr"), True),
}

rows, preds = [], {}
for name, m in models.items():
    m.fit(X.iloc[tr], y[tr])
    p = m.predict(X.iloc[te])
    assert np.isfinite(p).all()
    preds[name] = p
    rows.append({
        "Mô hình": name,
        "MAE (tỷ VND)": mean_absolute_error(y[te], p) / 1e9,
        "RMSE (tỷ VND)": np.sqrt(mean_squared_error(y[te], p)) / 1e9,
        "MdAPE (%)": 100 * np.median(np.abs(y[te] - p) / y[te]),
        "R² (giá gốc)": r2_score(y[te], p),
        "Số dự đoán âm": int((p < 0).sum()),
    })

metrics = pd.DataFrame(rows).set_index("Mô hình")
metrics.round(3).to_csv(OUT / "metrics.csv")
display(metrics.round(3))
print(f"\nTrung vị giá test: {np.median(y[te])/1e9:.2f} tỷ | trung bình: {y[te].mean()/1e9:.2f} tỷ")

### 4.1 Sai số của Ridge log — mô hình chọn trước

In [ ]:
best = "Ridge + log(target)"  # chọn trước, không chọn mô hình bằng test
p = preds[best]
resid = np.log1p(y[te]) - np.log1p(np.clip(p, 1, None))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y[te] / 1e9, p / 1e9, s=7, alpha=.3, color="#2f6fed")
lim = [0, np.percentile(y[te] / 1e9, 99.5)]
ax[0].plot(lim, lim, "r--", lw=1)
ax[0].set(title=f"{best}: thực tế vs dự đoán", xlabel="Giá thực (tỷ VND)", ylabel="Dự đoán (tỷ VND)")

ax[1].hist(resid, bins=60, color="#ff6a41")
ax[1].axvline(0, color="k", lw=1)
ax[1].set(title="Phân phối sai số (thang log)", xlabel="log(giá thực) − log(dự đoán)")

plt.tight_layout()
plt.savefig(OUT / "fig4_residuals.png", dpi=130, bbox_inches="tight")
plt.show()

# sai số theo phân khúc giá
band = pd.qcut(y[te], min(4, len(te)), duplicates="drop")
seg = pd.DataFrame({"band": band, "ape": np.abs(y[te] - p) / y[te] * 100}).groupby("band", observed=True).ape.median()
print("MdAPE (%) theo phân khúc giá:")
print(seg.round(1).to_string())

### 4.2 Hệ số — biến nào đẩy giá lên

In [ ]:
lin = models["Ridge + log(target)"].regressor_.named_steps["model"]
feat = models["Ridge + log(target)"].regressor_.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lin.coef_, index=feat).sort_values()

print("Hệ số Ridge log, sau chuẩn hóa cột số:")
print(pd.concat([coef.head(8), coef.tail(8)]).round(3).to_string())
print("\nKhông suy diễn nhân quả. Cột số đã chuẩn hóa; one-hot đầy đủ, không bỏ nhóm gốc.\nVới cùng biến phân loại, so sánh chênh lệch giữa hai hệ số thay vì coi một hệ số là mức tăng so nhóm gốc.")

## 5. Kết luận và giới hạn

**Viết trong báo cáo:**

1. Dữ liệu là **giá rao bán**, không phải giá giao dịch — mọi kết luận chỉ nói về hành vi định giá của người đăng tin.
2. `price` có giá trị cực đoan cần kiểm tra khả năng sai đơn vị; đã lọc theo phạm vi diện tích 10–500 m² và giá dương, nhưng **chưa xác minh từng dòng**.
3. Chia tập theo nhóm tiêu đề chỉ là proxy chống trùng; tin sửa tiêu đề vẫn có thể rò rỉ giữa train và test.
4. Không dùng `price_per_m2` làm feature (nó là hàm của target). Đã kiểm tra điều này bằng assert bên dưới.
5. So với baseline trên các chỉ số thật; không khẳng định Ridge tốt hơn trước khi chạy. Nếu cần tune, dùng group cross-validation trên train, không tune bằng test.
6. Kiểm tra sai số theo phân khúc; nếu cao ở nhóm nhà đắt, xem lại nhiễu giá, độ đại diện và đặc trưng. Chưa đủ dữ kiện khẳng định nguyên nhân.
7. Split nhóm tiêu đề không phải kiểm định dự báo tương lai. Kết quả là ước lượng trong phạm vi dataset đã lọc.

In [ ]:
# kiểm tra chống rò rỉ
assert not {"price", "price_per_m2", "log_price"} & set(NUM + CAT)
assert np.isfinite(df.price).all() and (df.price > 0).all()
import joblib
joblib.dump(models["Ridge + log(target)"], OUT / "ridge_log.joblib")
state = {"evaluation": "SMOKE_TEST_ONLY" if len(df) < 1000 else "BASELINE_NOT_TUNED",
         "train_rows": len(tr), "test_rows": len(te), "seed": 42,
         "target": "asking_price_VND", "note": "Extreme prices not individually verified"}
(OUT / "run.json").write_text(json.dumps(state, ensure_ascii=False, indent=2))

print("Đã ghi:")
for f in sorted(OUT.iterdir()):
    print(" -", f.name, f"{f.stat().st_size/1024:.0f} KB")